In [1]:
#adding sql file
import duckdb
from pathlib import Path
import re


In [2]:
db_path = Path("/Volumes/DATA/logist/notebooks/SQL/logistics.duckdb")
#db_path.parent.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(db_path))
print(db_path)

cpath=Path("/Volumes/DATA/logist/data/processed/cleaned_oper.csv")
print(cpath.exists())

con.execute(""" CREATE TABLE operations AS 
SELECT * FROM read_csv_auto(?)
""",[str(cpath)])

/Volumes/DATA/logist/notebooks/SQL/logistics.duckdb
True


test=duckdb.connect(str(db_path))
test.execute("SHOW TABLES").fetchdf()

con.close()



In [3]:
#eXEC SQL FILES
sql_file=Path("/Volumes/DATA/logist/notebooks/SQL/oper_analysis.sql")
def get_query(que_name):
    sql_txt=sql_file.read_text()
    pattern = ( 
        r"-- QUERY: \s*" + re.escape(que_name) + r"\s*(.*?)"
        r"(?=--\s*QUERY:|\Z)"
    )
    match= re.search(pattern,sql_txt,re.DOTALL | re.IGNORECASE)
    if not match:
        raise ValueError (f"Query '{que_name}' not found ")
    return match.group(1).strip()
    
def run_que(query):
    q=get_query(query)
    return con.execute(q).fetchdf()


**1.Warehouses with most volume, wrt operational performance**

In [4]:
run_que("WAREHOUSE PERFORMANCE")

,warehouse_id,total_ships,total_backlog,backlog_rate_percet,avg__util,avg_produc,avg_on_time_deliv,avg_error_percet
0,WH02,13458032.0,173104.0,1.29,65.85,7.14,95.56,0.85
1,WH01,11274144.0,196564.0,1.74,68.62,7.04,95.54,0.85
2,WH04,10200656.0,219093.0,2.15,70.78,6.99,95.41,0.86
3,WH03,9411536.0,439460.0,4.67,81.89,7.40,94.78,0.93
4,WH05,8035696.0,378520.0,4.71,82.31,6.96,94.84,0.95


**2.Warehouses with most utilization, and high backlog**

In [5]:
run_que("RISKY OPERATIONAL WAREHOUSES")

,warehouse_id,avg_util,total_backlog,backlog_rate_percet,avg_on_time_deliv
0,WH05,82.31,378520.0,4.71,94.84
1,WH03,81.89,439460.0,4.67,94.78
2,WH04,70.78,219093.0,2.15,95.41


**3.Warehouses rank wrt with backlog rate**

In [6]:
run_que("WAREHOUSES WRT BACKLOG RATE")

,warehouse_id,backlog_rate_percet,backlog_risk_rank
0,WH05,4.71,1
1,WH03,4.67,2
2,WH04,2.15,3
3,WH01,1.74,4
4,WH02,1.29,5


**4.Shift Performance best and worse**

In [7]:
run_que("SHIFT PERFORMANCE")

,shift_id,avg_produc,avg_util,total_backlog,avg_error_rate,avg_on_time_deliv
0,S1,7.71,73.64,541096.0,0.89,95.29
1,S2,7.14,71.66,424760.0,0.88,95.27
2,S3,6.48,76.37,440885.0,0.90,95.11


**5.warehouse/shift combinations are struggling wrt backlogs**

In [8]:
run_que("WAREHOUSE AND SHIFTS STRUGGLING")

,warehouse_id,shift_id,avg_produc,avg_util,total_backlog,avg_error_rate,avg_on_time_deliv
0,WH03,S1,8.04,81.70,170192.0,0.94,94.87
1,WH05,S1,7.57,82.38,143720.0,0.94,94.89
2,WH03,S3,6.80,85.34,141196.0,0.95,94.55
3,WH05,S2,7.14,81.59,132744.0,0.94,94.74
4,WH03,S2,7.37,78.61,128072.0,0.91,94.92
5,WH05,S3,6.17,82.96,102056.0,0.96,94.88
6,WH04,S3,6.48,74.41,84829.0,0.88,95.18
7,WH04,S1,7.55,70.18,84392.0,0.85,95.53
8,WH01,S1,7.70,68.99,73480.0,0.86,95.61
9,WH02,S1,7.67,64.96,69312.0,0.84,95.56


**6.days had the highest backlog**

In [9]:
run_que("WORST OPERATION DAY")

,date,warehouse_id,shift_id,shipments,capacity,utilization_perct,backlog,on_time_ships_percet,errors_percet
0,2025-12-04,WH02,S2,30160,20160,149.60,10000,89.73,0.91
1,2025-12-05,WH03,S1,21256,12600,168.70,8656,87.52,1.13
2,2025-12-12,WH01,S1,26488,17850,148.39,8638,88.49,0.85
3,2025-12-12,WH02,S1,29832,22680,131.53,7152,88.00,1.12
4,2025-12-04,WH03,S1,19744,12600,156.70,7144,88.03,1.03
5,2025-12-12,WH05,S2,16512,9520,173.45,6992,88.69,1.20
6,2025-12-19,WH03,S1,19576,12600,155.37,6976,88.80,1.07
7,2025-12-08,WH03,S1,19528,12600,154.98,6928,88.64,1.14
8,2025-12-12,WH03,S1,19456,12600,154.41,6856,87.84,0.90
9,2025-12-29,WH03,S1,19456,12600,154.41,6856,87.83,1.04


**7.highest utilized periods operating closest to or above capacity**

In [10]:
run_que("UTILIZATION PERIOD")

,date,warehouse_id,shift_id,shipments,capacity,utilization_perct,backlog,on_time_ships_percet
0,2025-12-12,WH05,S3,12392,6664,185.95,5728,87.53
1,2025-12-25,WH03,S3,13576,7650,177.46,5926,88.12
2,2025-12-12,WH05,S2,16512,9520,173.45,6992,88.69
3,2025-12-30,WH05,S3,11544,6664,173.23,4880,88.43
4,2025-12-10,WH03,S3,13096,7650,171.19,5446,89.94
5,2025-12-05,WH03,S1,21256,12600,168.70,8656,87.52
6,2025-12-26,WH03,S3,12872,7650,168.26,5222,87.37
7,2025-12-19,WH05,S1,17280,10584,163.27,6696,88.00
8,2025-12-09,WH03,S3,12416,7650,162.30,4766,87.46
9,2025-12-18,WH03,S3,12408,7650,162.20,4758,89.27


**8.Performance at peak and off perak**

In [11]:
run_que("PEAK PERFORMANCE")

,is_peak,periods,total_shipments,avg_util,total_backlog,avg_error_rate,avg_on_time_deliv
0,False,3948,34131328.0,62.42,0.0,0.80,96.03
1,True,1527,18248736.0,103.54,1406741.0,1.11,93.15


**9.Performance throughout the year wrt monthly**


In [12]:
run_que("MONLTHY PERFORMANCE")

,month_nm,periods,total_shipments,avg_util,total_backlog,avg_error_rate,avg_on_time_deliv
0,January,465,3809576.0,63.19,0.0,0.82,96.03
1,February,420,3186288.0,58.60,0.0,0.80,96.01
2,March,465,3916384.0,65.04,1458.0,0.84,95.97
3,April,450,4088840.0,70.12,4891.0,0.87,95.91
4,May,465,4420784.0,73.36,16500.0,0.89,95.77
5,June,450,4250160.0,73.15,18232.0,0.89,95.72
6,July,465,4019880.0,66.76,2446.0,0.85,95.91
7,August,465,3718560.0,61.76,0.0,0.83,96.02
8,September,450,4292728.0,73.63,19787.0,0.89,95.69
9,October,465,4712112.0,78.11,38957.0,0.93,95.41


**10. Operational pressure calc. wrt day of the week**

In [13]:
run_que("DAILY PERFORMACE")

,day_of_week,periods,total_shipments,avg_util,total_backlog,avg_error_rate,avg_on_time_deliv
0,Friday,780,8892856.0,88.19,382766.0,0.95,94.64
1,Monday,780,8617832.0,85.24,291319.0,0.93,94.78
2,Thursday,780,8537848.0,84.44,262394.0,0.92,94.98
3,Wednesday,795,8384432.0,81.36,233769.0,0.90,95.08
4,Tuesday,780,8173304.0,81.07,236015.0,0.90,95.10
5,Saturday,780,5278400.0,52.27,478.0,0.81,96.02
6,Sunday,780,4495392.0,44.51,0.0,0.81,95.99


**11.operational periods officials investigate first**

In [14]:
run_que("HIGH PRESSURE PERIODS")

,date,warehouse_id,shift_id,shipments,capacity,utilization_perct,backlog,errors_percet,on_time_ships_percet
0,2025-12-12,WH05,S3,12392,6664,185.95,5728,1.13,87.53
1,2025-12-25,WH03,S3,13576,7650,177.46,5926,1.07,88.12
2,2025-12-12,WH05,S2,16512,9520,173.45,6992,1.20,88.69
3,2025-12-30,WH05,S3,11544,6664,173.23,4880,1.16,88.43
4,2025-12-10,WH03,S3,13096,7650,171.19,5446,1.07,89.94
...,...,...,...,...,...,...,...,...,...
671,2025-10-22,WH05,S3,6672,6664,100.12,8,0.70,95.66
672,2025-05-29,WH04,S2,14448,14432,100.11,16,1.30,97.31
673,2025-12-22,WH02,S2,20176,20160,100.08,16,0.85,97.49
674,2025-05-05,WH03,S2,11408,11400,100.07,8,1.01,96.63


**12. warehouses experience high-pressure conditions most frequently**

In [15]:
run_que("FREQ HIGH PRESSURE WAREHOUSES")

,warehouse_id,total_periods,high_pressure_periods,high_pressure_perct
0,WH03,1095,211.0,19.27
1,WH05,1095,205.0,18.72
2,WH04,1095,100.0,9.13
3,WH01,1095,87.0,7.95
4,WH02,1095,73.0,6.67


**13. periods that had poor service performance**

In [16]:
run_que("PERIODS WITH POOR PERFORMANCE")

,date,warehouse_id,shift_id,shipments,utilization_perct,backlog,on_time_ships_percet
0,2025-12-05,WH04,S2,16856,116.80,2424,85.06
1,2025-12-26,WH05,S2,12936,135.88,3416,85.52
2,2025-12-23,WH05,S3,8160,122.45,1496,85.86
3,2025-12-03,WH05,S3,8784,131.81,2120,86.13
4,2025-11-03,WH03,S2,13328,116.91,1928,86.17
5,2025-12-12,WH04,S2,18016,124.83,3584,86.18
6,2025-12-03,WH01,S2,22688,139.02,6368,86.32
7,2025-12-09,WH01,S2,19800,121.32,3480,86.38
8,2025-12-22,WH05,S2,11336,119.08,1816,86.38
9,2025-12-11,WH04,S3,12560,132.50,3081,86.40


**14. periods had unusually high error rates**

In [17]:
run_que("UNUSUAL HIGH ERROR RATES")

,date,warehouse_id,shift_id,shipments,utilization_perct,errors_percet,backlog,on_time_ships_percet
0,2025-12-25,WH05,S3,9392,140.94,1.74,2728,88.24
1,2025-10-23,WH05,S3,6816,102.28,1.67,152,96.25
2,2025-06-26,WH05,S1,10232,96.67,1.64,0,96.46
3,2025-09-26,WH04,S1,14896,94.03,1.63,0,96.99
4,2025-05-12,WH02,S3,12048,89.48,1.54,0,97.61
5,2025-12-12,WH04,S3,13664,144.15,1.53,4185,87.45
6,2025-04-16,WH03,S1,11464,90.98,1.53,0,96.30
7,2025-06-23,WH03,S2,10672,93.61,1.53,0,97.60
8,2025-11-03,WH05,S1,10448,98.72,1.53,0,95.87
9,2025-09-04,WH03,S3,7976,104.26,1.53,326,91.38


**15. warehouse/shift combinations have the strongest productivity**

In [18]:
run_que("PRODUCTIVITY WRT WAREHOUSE/SHIFTS")

,warehouse_id,shift_id,avg_produc,avg_util,avg_on_time_deliv
0,WH03,S1,8.04,81.70,94.87
1,WH01,S1,7.70,68.99,95.61
2,WH02,S1,7.67,64.96,95.56
3,WH05,S1,7.57,82.38,94.89
4,WH04,S1,7.55,70.18,95.53
5,WH03,S2,7.37,78.61,94.92
6,WH02,S2,7.20,63.99,95.58
7,WH05,S2,7.14,81.59,94.74
8,WH01,S2,7.05,66.37,95.59
9,WH04,S2,6.94,67.73,95.52


**16. warehouse performing better or worse than the overall network**

In [19]:
run_que("WAREHOUSE WRT AVG NETWORK PERFORMANCE")

,warehouse_id,avg_util,utilization_vs_network,avg_on_time_deliv,on_time_vs_network,avg_error_rate
0,WH05,82.31,8.42,94.84,-0.39,0.95
1,WH03,81.89,8.00,94.78,-0.45,0.93
2,WH04,70.78,-3.11,95.41,0.18,0.86
3,WH01,68.62,-5.27,95.54,0.31,0.85
4,WH02,65.85,-8.04,95.56,0.34,0.85


**17. three worst backlog periods for every warehouse**

In [20]:
run_que("TOP 3 WORST BACKLOG PERIODS WRT WAREHOUSES")

,date,warehouse_id,shift_id,shipments,utilization_perct,backlog,on_time_ships_percet,period_rank
0,2025-12-12,WH01,S1,26488,148.39,8638,88.49,1
1,2025-12-22,WH01,S1,24528,137.41,6678,88.00,2
2,2025-12-03,WH01,S2,22688,139.02,6368,86.32,3
3,2025-12-04,WH02,S2,30160,149.60,10000,89.73,1
4,2025-12-12,WH02,S1,29832,131.53,7152,88.00,2
5,2025-12-26,WH02,S1,28848,127.20,6168,88.31,3
6,2025-12-05,WH03,S1,21256,168.70,8656,87.52,1
7,2025-12-04,WH03,S1,19744,156.70,7144,88.03,2
8,2025-12-19,WH03,S1,19576,155.37,6976,88.80,3
9,2025-12-01,WH04,S1,22304,140.79,6462,86.97,1


**18. Final mgmt exception**

In [21]:
run_que("FINAL MGMT EXCEPTION REPORT")

,date,warehouse_id,shift_id,shipments,capacity,utilization_perct,backlog,errors_percet,on_time_ships_percet,operational_risk
0,2025-12-04,WH02,S2,30160,20160,149.60,10000,0.91,89.73,CRITICAL
1,2025-12-05,WH03,S1,21256,12600,168.70,8656,1.13,87.52,CRITICAL
2,2025-12-12,WH01,S1,26488,17850,148.39,8638,0.85,88.49,CRITICAL
3,2025-12-12,WH02,S1,29832,22680,131.53,7152,1.12,88.00,CRITICAL
4,2025-12-04,WH03,S1,19744,12600,156.70,7144,1.03,88.03,CRITICAL
...,...,...,...,...,...,...,...,...,...,...
5470,2025-03-23,WH03,S3,3960,7650,51.76,0,0.78,94.61,NORMAL
5471,2025-03-23,WH04,S3,4144,9479,43.72,0,0.70,97.88,NORMAL
5472,2025-05-11,WH01,S2,6856,16320,42.01,0,0.96,96.69,NORMAL
5473,2025-03-24,WH01,S2,11288,16320,69.17,0,0.63,96.12,NORMAL


We want a small set of meaningful datasets:

1. Warehouse KPI
2. Shift KPI
3. Monthly KPI
4. Daily/period KPI
5. High-pressure periods

In [25]:
warehouse_kpi=run_que("warehouse_kpi")
warehouse_kpi.to_csv("/Volumes/DATA/logist/data/processed/warehouse_kpi.csv",index=False)

In [28]:
shift_kpi=run_que("shift_kpi")
shift_kpi.to_csv("/Volumes/DATA/logist/data/processed/shift_kpi.csv",index=False)

In [30]:
monthly_kpi=run_que("monthly_kpi")
monthly_kpi.to_csv("/Volumes/DATA/logist/data/processed/monthly_kpi.csv",index=False)

In [31]:
high_pressure_by_warehouse=run_que("high_pressure_by_warehouse")
high_pressure_by_warehouse.to_csv("/Volumes/DATA/logist/data/processed/high_pressure_by_warehouse.csv",index=False)